# pymefpp_create_associations

This notebook is used to create the associations required to run mefpp


# Testing notebook

It works without any problem or warning

In [5]:
import os
import sys
import glob
import logging

from pathlib import Path

# here we import the local mefpp_run library
sys.path.insert(0, os.getcwd())
import mefpp_associate_reduction as mar

import pycavi_tools
import pycavi_instrument

In [6]:
# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# debug_mode: True
# log_level: info
# observatory: Paranal
# telescope: VST
# instrument: OmegaCam
# file_prefix: OMEGA
# raw_data_dir: /Users/fselman/Pictures/OCAM/CPPup/raw
# project_dir: /Users/fselman/Pictures/OCAM/CPPup2
# reduced_calib_dir: /Users/fselman/Pictures/OCAM/CPPup2/calib/
# reduced_science_dir: /Users/fselman/Pictures/OCAM/CPPup2/science/
# project_logs_dir: /Users/fselman/Pictures/OCAM/CPPup2/logs/
# sex_dir_name: Sextractor_dir
# scamp_dir_name: Scamp_dir
# swarp_dir_name: Swarp_dir
# cal_dir_name: Calibration_dir
#
config_dir = os.getcwd()+'/config/'
config = pycavi_tools.read_pycavi_config(config_dir)
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']

# Should get these from an instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir, instrument='OCAM')
bpm_prefix = instrument.bpm_prefix
bias_prefix = instrument.bias_prefix
domeff_prefix = instrument.domeff_prefix
skyff_prefix = instrument.skyff_prefix
mff_prefix = instrument.mff_prefix

#create the log_dir outside the science dir.
try:
    os.makedirs(log_dir, exist_ok=True)
except Exception as e:
    print(e)
    pass    


# we prepare the logs
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s:%(levelname)s:%(message)s",
                    handlers=[
                              logging.FileHandler(log_dir+"/pymeffpp_associate.log"),
                              logging.StreamHandler()
                    ])

MEF.BIAS.NAME.O: OCAM_MB
MEF.BPM.NAME.O: OCAM_BPM
MEF.DOMEFF.NAME.O: OCAM_DOMEFF
MEF.SKYFF.NAME.O: OCAM_SKYFF
MEF.MFF.NAME.O: OCAM_MFF


In [7]:
def link_bpm(partitions, bpm_partition, instrument, part_type='sci'):
    """
    Creates symbolic links in all partitions in list partitions, to the
    corresponding master BPM file in the static calibration directory.

    Note that the BPM files needs to exists before creating the symbolic links.
    """
    for part in partitions:
        part.bpmPartition = bpm_partition
        #print((part.bpmPartition).base_path)
        #print(instrument.bpm_prefix+'*'+instrument.reduced_file_suffix)
        for i in range(1, instrument.nextensions+1):
            bpm_file = (part.bpmPartition).base_path+'/'+instrument.bpm_prefix+'_%02d.fits'%(i,)
            local_bpm_filename = part.base_path+'/'+instrument.bpm_prefix+'_%02d.fits'%(i,)
            if not Path(bpm_file).is_file():
                try:
                    os.symlink(bpm_file, local_bpm_filename)
                    print("create symbolic link from : {}".format(local_bpm_filename))
                    print("                       to : {}".format(bpm_file))
                    Path((part.bpmPartition).base_path+'/toreduce_'+part_type).touch()
                    pass
                except FileExistsError:
                    pass
                except Exception as e:
                    print(e)
                    pass
                
            else:
                print("symbolic link exists from : {}".format(local_bpm_filename))
                print("                       to : {}".format(bpm_file))

def select_nearest_calib(partitions, cal_partitions, calpart_type='bias', ndays_max=10):
    """
    Select the calibration nearest in time. For dome, sky flats, std, and sci we
    match the filter.

    This function does the linking only in memory. The command link needs to be
    invoqued afterward.
    """
    for part in partitions:
        # here we look for the closest in time bias partition
        zdate = part.OBstart

        # we find the bias closest in time
        calBestDeltaT = 3600.*24.*ndays_max
        for cal_part in cal_partitions:
            if 'bias' not in calpart_type:
                if part.filter != cal_part.filter:
                    continue
            cal_date = cal_part.OBstart
            deltaT = mar.timeDistance(zdate, cal_date)
            #print(sci_date, bias_date, deltaT)
            if deltaT < calBestDeltaT:
                calBestDeltaT = deltaT
                if 'bias' in calpart_type:
                    part.biasPartition = cal_part
                elif 'dome' in calpart_type:
                    part.domePartition = cal_part
                elif 'sky' in calpart_type:
                    part.skyPartition = cal_part
                    part.mffPartition = cal_part
            if hasattr(cal_part, 'filter'):
                print(f'partition: {part.filter} {part.OBstart}\n    {calpart_type}: {cal_part.OBstart} {cal_part.filter} {round(calBestDeltaT/3600.0/24., 3)}')
            else:
                print(f'partition: {part.filter} {part.OBstart}\n    {calpart_type}: {cal_part.OBstart} {round(calBestDeltaT/3600.0/24., 3)}')

def link_virtual_products(partitions, partition_type='sci', calpart_type='bias', instrument=instrument):

    for part in partitions:
        if 'bias' in calpart_type:
            prefix = instrument.bias_prefix
            base_path = (part.biasPartition).base_path
            mcal = (part.biasPartition).base_path+'/'+prefix+'_1x1.fits'
            mcal_link = part.base_path+'/'+prefix+'_1x1.fits'
        if 'mff' in calpart_type:
            prefix = instrument.mff_prefix
            base_path = (part.mffPartition).base_path
            mcal = (part.mffPartition).base_path+'/'+prefix+'_'+part.filter+'_1x1.fits'
            mcal_link = part.base_path+'/'+prefix+'_'+part.filter+'_1x1.fits'
        if 'dome' in calpart_type:
            prefix = instrument.domeff_prefix
            base_path = (part.domePartition).base_path
            mcal = (part.domePartition).base_path+'/'+prefix+'_'+part.filter+'_1x1.fits'
            mcal_link = part.base_path+'/'+prefix+'_'+part.filter+'_1x1.fits'

        if not Path(mcal_link).is_symlink():
            try:
                os.symlink(mcal, mcal_link)
            except FileExistsError:
                pass
            except Exception as e:
                print(e)
                pass
            Path(base_path+'/toreduce_'+partition_type).touch()
            print(mcal)
        

In [8]:
exposurePartition = mar.PartitionsTree(raw_data_dir, static_cal_dir, reduced_cal_dir, reduced_science_dir, log_dir, instrument)
bpm_partitions = exposurePartition.bpm_partitions
bias_partitions = exposurePartition.bias_partitions
dome_partitions = exposurePartition.dome_partitions
sky_partitions = exposurePartition.sky_partitions
std_partitions = exposurePartition.std_partitions
sci_partitions = exposurePartition.sci_partitions

argument for glob.glob: /Users/fselman/Pictures/OCAM/m67_raw//**/OMEGA.*.fits
number of raw files found: 83

BIAS:     2 DOME:     1 SKY:      2 STD:      0 SCIENCE:  1 Total     6
BIAS:     3 DOME:     1 SKY:      2 STD:      1 SCIENCE:  1 Total     8
BIAS:     3 DOME:     1 SKY:      2 STD:      1 SCIENCE:  1 Total     8
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  1 Total     9
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  1 Total     9
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  1 Total     9
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  2 Total     10
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  2 Total     10
BIAS:     3 DOME:     2 SKY:      2 STD:      1 SCIENCE:  3 Total     11
BIAS:     3 DOME:     2 SKY:      3 STD:      1 SCIENCE:  3 Total     12
BIAS:     3 DOME:     2 SKY:      3 STD:      1 SCIENCE:  3 Total     12
BIAS:     3 DOME:     2 SKY:      3 STD:      1 SCIENCE:  3 Total     12
BIAS:     3 DOME:    

2024-07-01 10:58:47,003:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:48:12.247.fits
2024-07-01 10:58:47,004:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:45:23.427.fits
2024-07-01 10:58:47,006:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:43:59.207.fits
2024-07-01 10:58:47,007:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:44:41.317.fits
2024-07-01 10:58:47,009:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:48:57.618.fits
2024-07-01 10:58:47,010:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:46:05.538.fits
2024-07-01 10:58:47,011:INFO:bias/574184/2014-12-02T08:35:57 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T08:43:17.096.fits
2024-07-01 10:58:47,012:INFO:bias/574184/2014-12

BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14
BIAS:     3 DOME:     3 SKY:      3 STD:      2 SCIENCE:  3 Total     14


2024-07-01 10:58:47,093:INFO:sky/200253598/NB_659/2014-12-02T23:28:39 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T23:30:21.845.fits
2024-07-01 10:58:47,096:INFO:sky/200253598/NB_659/2014-12-02T23:28:39 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-12-02T23:32:45.405.fits
2024-07-01 10:58:47,099:INFO:sky/200254042/g_SDSS/2014-11-30T23:28:02 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-11-30T23:32:17.576.fits
2024-07-01 10:58:47,101:INFO:sky/200254042/g_SDSS/2014-11-30T23:28:02 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-11-30T23:33:02.736.fits
2024-07-01 10:58:47,104:INFO:sky/200254042/g_SDSS/2014-11-30T23:28:02 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-11-30T23:30:47.716.fits
2024-07-01 10:58:47,105:INFO:sky/200254042/g_SDSS/2014-11-30T23:28:02 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-11-30T23:30:03.016.fits
2024-07-01 10:58:47,107:INFO:sky/200254042/g_SDSS/2014-11-30T23:28:02 -- /Users/fselman/Pictures/OCAM/m67_raw/OMEGA.2014-11-30T23:

number of bias partitions:  3
2014-12-02T08:35:57
2014-12-01T09:02:04
2014-12-03T08:25:12
number of dome partitions:  3
2014-12-02T08:35:57 NB_659
2014-12-02T08:35:57 r_SDSS
2014-12-02T08:35:57 g_SDSS
number of sky partitions:  3
2014-11-30T23:28:02 g_SDSS
2014-12-02T23:40:05 r_SDSS
2014-12-02T23:28:39 NB_659
number of standard stars partitions:  3
2014-12-02T05:57:25 r_SDSS
2014-12-02T05:57:25 g_SDSS
2014-12-02T07:31:14 NB_659
number of science partitions:  3
2014-12-02T07:04:51 NB_659
2014-12-02T06:51:24 r_SDSS
2014-12-02T06:32:53 g_SDSS


NameError: name 'mar' is not defined

In [ ]:
# We associate a bpm partition to each science frame and create symbolic link in
# corresponding directory
link_bpm(sci_partitions, bpm_partitions[0], instrument, part_type='sci')
link_bpm(dome_partitions, bpm_partitions[0], instrument, part_type='dome')
link_bpm(sky_partitions, bpm_partitions[0], instrument, part_type='sky')
link_bpm(std_partitions, bpm_partitions[0], instrument, part_type='std')
link_bpm(bias_partitions, bpm_partitions[0], instrument, part_type='bias')

In [ ]:
# The following operations are done only on memory.

# select and attach bias partition to science partitions
select_nearest_calib(sci_partitions, bias_partitions, calpart_type='bias', ndays_max=10)
select_nearest_calib(std_partitions, bias_partitions, calpart_type='bias', ndays_max=10)

# select and attach sky partition to science partitions
select_nearest_calib(sci_partitions, sky_partitions, calpart_type='sky', ndays_max=30)   # Note that this also links the master flat
select_nearest_calib(std_partitions, sky_partitions, calpart_type='sky', ndays_max=30)   # Note that this also links the master flat

# select and attach bias and dome partitions to sky partitions
select_nearest_calib(sky_partitions, bias_partitions, calpart_type='bias', ndays_max=30)
select_nearest_calib(sky_partitions, dome_partitions, calpart_type='dome', ndays_max=30)

# select and attach bias partitions to dome partitions
select_nearest_calib(dome_partitions, bias_partitions, calpart_type='bias', ndays_max=30)


In [ ]:
# at this point we have the biases and mff for each science frame.
# Now we create virtual calibration products as a symbolic link.
#
link_virtual_products(sci_partitions, partition_type='sci', calpart_type='bias', instrument=instrument)
link_virtual_products(std_partitions, partition_type='std', calpart_type='bias', instrument=instrument)

link_virtual_products(sci_partitions, partition_type='sci', calpart_type='mff', instrument=instrument)
link_virtual_products(std_partitions, partition_type='std', calpart_type='mff', instrument=instrument)

link_virtual_products(sky_partitions, partition_type='sky', calpart_type='bias', instrument=instrument)
link_virtual_products(sky_partitions, partition_type='sky', calpart_type='dome', instrument=instrument)
link_virtual_products(dome_partitions, partition_type='dome', calpart_type='bias', instrument=instrument)